# InSight — Sephora Cosmetics Dataset Exploration & Preparation

This notebook performs exploratory data analysis (EDA), data validation, text cleaning, weak sentiment labeling, and ML-ready sampling for the Sephora cosmetics customer review dataset.

---

## Google Colab Execution Instructions

This notebook is configured to run smoothly in **both Google Colab and local environments**.

### Option A: Running with Google Drive (Recommended for persistent storage)
1. Upload the `InSight_ML` directory to your Google Drive under **`My Drive`** (so the path is `My Drive/InSight_ML`).
2. Alternatively, upload `archive.zip` to your Google Drive root or inside `InSight_ML/data/raw/cosmetics/`.
3. Run **Section 1** below. When prompted, authorize Google Drive access.
4. The path resolver will automatically detect your Drive folder.

### Option B: Running without Google Drive (Colab session storage)
1. In Google Colab, click the **Files** icon (folder icon on the left sidebar).
2. Drag and drop `archive.zip` directly into the `/content` folder.
3. Run the notebook — the dynamic path resolver will automatically detect `/content/archive.zip` and run without requiring Drive.

### Option C: Running Locally in VS Code / JupyterLab
1. No setup required. The notebook automatically detects local workspace paths relative to `InSight_ML`.

In [ ]:
# =============================================================================
# 1. Environment Setup & Dynamic Path Resolution (Colab & Local)
# =============================================================================
import os
import sys
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Detect Colab environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running inside Google Colab environment.")
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print("Google Drive successfully mounted at /content/drive")
    except Exception as e:
        print("Google Drive mount skipped or already mounted:", e)

# Candidate locations for archive.zip (Drive, Colab root, or local repo)
candidate_paths = [
    "/content/drive/MyDrive/InSight_ML/data/raw/cosmetics/archive.zip",
    "/content/drive/MyDrive/archive.zip",
    "/content/archive.zip",
    "InSight_ML/data/raw/cosmetics/archive.zip",
    "../data/raw/cosmetics/archive.zip",
    "data/raw/cosmetics/archive.zip"
]

ZIP_PATH = None
for p in candidate_paths:
    if os.path.exists(p):
        ZIP_PATH = p
        break

if ZIP_PATH is None:
    raise FileNotFoundError(
        "archive.zip not found!\n"
        "Please ensure archive.zip is uploaded to Google Drive (/content/drive/MyDrive/InSight_ML/data/raw/cosmetics/archive.zip)\n"
        "or directly uploaded to Colab (/content/archive.zip)."
    )

# Resolve base output directories
if IN_COLAB and os.path.exists("/content/drive/MyDrive/InSight_ML"):
    BASE_DIR = "/content/drive/MyDrive/InSight_ML"
elif os.path.exists("InSight_ML"):
    BASE_DIR = "InSight_ML"
elif os.path.exists("../data"):
    BASE_DIR = ".."
else:
    BASE_DIR = "/content/InSight_ML" if IN_COLAB else "."

PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed", "cosmetics")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")
REPORTS_DIR = os.path.join(BASE_DIR, "reports")

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print("Using ZIP Archive :", ZIP_PATH)
print("Base ML Directory :", BASE_DIR)
print("Processed Dir     :", PROCESSED_DIR)
print("Outputs Dir       :", OUTPUTS_DIR)
print("Reports Dir       :", REPORTS_DIR)

## 2. Inspecting the ZIP Archive

Here we inspect the files inside `archive.zip` without extracting or uncompressing to disk, preserving storage.

In [ ]:
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    print("Files inside the ZIP archive:")
    for info in z.infolist():
        print(f"  - {info.filename:<22} ({info.file_size / 1e6:6.2f} MB uncompressed)")

In [ ]:
# Inspect columns and sample rows from each file inside the archive
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for filename in z.namelist():
        if filename.endswith(".csv"):
            print("\n" + "=" * 70)
            print("FILE:", filename)
            sample = pd.read_csv(z.open(filename), nrows=3)
            print(f"Columns ({len(sample.columns)}):", list(sample.columns))
            print("\nFirst 3 rows:")
            display(sample)

## 3. Product Catalog Exploration (`product_info.csv`)

The archive contains `product_info.csv` with catalog metadata across Sephora products (categories, brands, pricing, ingredients, ratings, loves_count).

In [ ]:
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    products_df = pd.read_csv(z.open("product_info.csv"))

print("Product Catalog Shape:", products_df.shape)
print("Unique Products       :", products_df["product_id"].nunique())
print("Unique Brands         :", products_df["brand_name"].nunique())

print("\nTop Primary Categories:")
print(products_df["primary_category"].value_counts())

print("\nPrice Statistics (USD):")
print(products_df["price_usd"].describe())

print("\nMissing values in product catalog:")
print(products_df[["product_id", "product_name", "brand_name", "price_usd", "primary_category", "ingredients"]].isnull().sum())

## 4. Memory-Efficient Review Data Loading & Validation

The archive contains over **1 million reviews** split across 5 partition files (`reviews_0-250.csv` to `reviews_1250-end.csv`).

**Memory-Efficient Strategy:**
1. Use `usecols` during `pd.read_csv()` to load only the required columns into memory.
2. Exclude personal demographic identifiers (`author_id`, `skin_tone`, `eye_color`, `skin_type`, `hair_color`) and row index artifacts (`Unnamed: 0`).
3. Drop null or whitespace-only review comments immediately upon loading each partition.
4. Validate data integrity: rating range (1-5), missing fields, duplicate entries, and submission date ranges.

In [ ]:
# Define columns to load (excluding demographic/author fields and row indices)
review_cols = [
    "product_id", "product_name", "brand_name", "price_usd",
    "rating", "is_recommended", "helpfulness", "total_feedback_count",
    "total_pos_feedback_count", "total_neg_feedback_count",
    "submission_time", "review_text", "review_title"
]

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    review_files = sorted([
        name for name in z.namelist()
        if name.startswith("reviews_") and name.endswith(".csv")
    ])
    
    chunks = []
    raw_counts = {}
    clean_counts = {}
    
    for name in review_files:
        chunk = pd.read_csv(z.open(name), usecols=review_cols, low_memory=False)
        raw_counts[name] = len(chunk)
        
        # Drop rows with null or whitespace-only review_text
        valid_chunk = chunk.dropna(subset=["review_text"])
        valid_chunk = valid_chunk[valid_chunk["review_text"].astype(str).str.strip().ne("")].copy()
        clean_counts[name] = len(valid_chunk)
        chunks.append(valid_chunk)
        print(f"Loaded {name:<20}: {raw_counts[name]:>7,} raw -> {clean_counts[name]:>7,} clean reviews")

reviews_all = pd.concat(chunks, ignore_index=True)
total_raw = sum(raw_counts.values())
total_clean = len(reviews_all)

print("\n" + "=" * 60)
print(f"Total raw reviews across files : {total_raw:,}")
print(f"Total clean valid reviews      : {total_clean:,} (dropped {total_raw - total_clean:,} empty texts)")
print(f"Memory footprint               : {reviews_all.memory_usage(deep=True).sum() / 1e6:.1f} MB")

# Validation Checks
print("\nMissing values in clean reviews:")
print(reviews_all.isnull().sum())

print("\nRating distribution across full dataset:")
print(reviews_all["rating"].value_counts().sort_index())

## 5. ML-Ready Sampling, Weak Sentiment Labeling & Catalog Linkage

In this step, we:
1. Draw a **reproducible sample of 10,000 reviews** (`random_state=42`).
2. Generate **weak sentiment labels** from star ratings:
   - **1–2 stars**: `negative`
   - **3 stars**: `neutral`
   - **4–5 stars**: `positive`
   *(Note: These are heuristic weak labels for baseline modeling, not verified ground truth).*
3. **Link to Product Catalog**: Join `primary_category`, `secondary_category`, `tertiary_category`, and `loves_count` from `product_info.csv` using `product_id`.
4. Compute text lengths and save the ML-ready dataset to `cosmetics_10k.csv`.

In [ ]:
# 1. Reproducible Random Sample of 10,000
sample_10k = reviews_all.sample(n=10000, random_state=42).reset_index(drop=True)

# 2. Weak Sentiment Labeling
def assign_weak_sentiment(r):
    if r in [1, 2]:
        return "negative"
    elif r == 3:
        return "neutral"
    elif r in [4, 5]:
        return "positive"
    return "unknown"

sample_10k["weak_sentiment"] = sample_10k["rating"].apply(assign_weak_sentiment)

# 3. Link with Product Catalog
product_meta = products_df[[
    "product_id", "primary_category", "secondary_category", "tertiary_category", "loves_count"
]].drop_duplicates(subset=["product_id"])

sample_10k = sample_10k.merge(product_meta, on="product_id", how="left")
linked_ratio = sample_10k["primary_category"].notna().mean()
print(f"Product catalog linkage rate: {linked_ratio:.1%}")

# 4. Word count and date conversion
sample_10k["submission_time"] = pd.to_datetime(sample_10k["submission_time"], errors="coerce")
sample_10k["word_count"] = sample_10k["review_text"].astype(str).str.split().str.len()

# 5. Save Processed Dataset
output_csv = os.path.join(PROCESSED_DIR, "cosmetics_10k.csv")
sample_10k.to_csv(output_csv, index=False)

print(f"\nSample successfully created and saved to: {output_csv}")
print(f"Dataset Shape: {sample_10k.shape}")
print("\nWeak Sentiment Distribution:")
print(sample_10k["weak_sentiment"].value_counts())

## 6. Inspecting the 10,000 Review Sample

Let's review the summary statistics, preview rows, and check data quality for the processed sample.

In [ ]:
print("Dataset Shape:", sample_10k.shape)

print("\nFirst 5 Reviews:")
display(sample_10k[["rating", "weak_sentiment", "product_name", "brand_name", "primary_category", "review_title", "word_count"]].head())

print("\nMissing values:")
print(sample_10k.isnull().sum())

print("\nWord count statistics:")
print(sample_10k["word_count"].describe())

print("\nDate range:")
print("Earliest:", sample_10k["submission_time"].min())
print("Latest  :", sample_10k["submission_time"].max())

## 7. Exploratory Visualizations

We generate visualizations for:
1. Star rating & weak sentiment distribution
2. Review word count distribution
3. Review trends over time
4. Top product categories and brands
5. Recommendation rate across rating tiers

In [ ]:
# 1. Rating & Weak Sentiment Distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
rating_counts = sample_10k["rating"].value_counts().sort_index()
colors_rating = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#27ae60"]
axes[0].bar(rating_counts.index, rating_counts.values, color=colors_rating, edgecolor="#333333", alpha=0.85)
axes[0].set_title("Rating Distribution (10k Sample)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Star Rating")
axes[0].set_ylabel("Review Count")
for x, y in zip(rating_counts.index, rating_counts.values):
    axes[0].text(x, y + 80, f"{y:,}\n({y/len(sample_10k):.1%})", ha="center", fontsize=9)
axes[0].set_ylim(0, max(rating_counts.values) * 1.15)

sentiment_order = ["negative", "neutral", "positive"]
sentiment_counts = sample_10k["weak_sentiment"].value_counts().reindex(sentiment_order)
colors_sentiment = ["#e74c3c", "#f39c12", "#2ecc71"]
axes[1].bar(sentiment_counts.index, sentiment_counts.values, color=colors_sentiment, edgecolor="#333333", alpha=0.85)
axes[1].set_title("Weak Sentiment Label Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Weak Sentiment Label")
axes[1].set_ylabel("Review Count")
for x, y in zip(sentiment_counts.index, sentiment_counts.values):
    axes[1].text(x, y + 80, f"{y:,}\n({y/len(sample_10k):.1%})", ha="center", fontsize=9)
axes[1].set_ylim(0, max(sentiment_counts.values) * 1.15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "rating_and_sentiment_distribution.png"), dpi=150)
plt.show()

# 2. Review Length Distribution
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(sample_10k["word_count"], bins=50, range=(0, 250), color="#3498db", edgecolor="#2980b9", alpha=0.85)
ax.set_title("Distribution of Review Word Counts (Capped at 250 words)", fontsize=13, fontweight="bold")
ax.set_xlabel("Word Count")
ax.set_ylabel("Number of Reviews")
median_len = sample_10k["word_count"].median()
mean_len = sample_10k["word_count"].mean()
ax.axvline(median_len, color="#e74c3c", linestyle="--", label=f"Median: {median_len:.0f} words")
ax.axvline(mean_len, color="#2ecc71", linestyle="-", label=f"Mean: {mean_len:.1f} words")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "review_length_distribution.png"), dpi=150)
plt.show()

# 3. Reviews Over Time
yearly = (
    sample_10k.dropna(subset=["submission_time"])
    .set_index("submission_time")
    .resample("YE")
    .size()
)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(yearly.index.year, yearly.values, marker="o", color="#8e44ad", linewidth=2.2, markersize=6)
ax.set_title("Review Volume by Year (10k Sample)", fontsize=13, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Number of Reviews")
ax.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "reviews_trend_over_time.png"), dpi=150)
plt.show()

# 4. Top Categories & Brands
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top_cats = sample_10k["primary_category"].value_counts().head(8)
axes[0].barh(top_cats.index[::-1], top_cats.values[::-1], color="#1abc9c", edgecolor="#16a085")
axes[0].set_title("Top Product Categories (10k Sample)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Review Count")

top_brands = sample_10k["brand_name"].value_counts().head(10)
axes[1].barh(top_brands.index[::-1], top_brands.values[::-1], color="#34495e", edgecolor="#2c3e50")
axes[1].set_title("Top Reviewed Brands (10k Sample)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Review Count")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "top_categories_and_brands.png"), dpi=150)
plt.show()

## 8. Common Words & Topical Vocabulary

Analyzing the most frequent words in customer reviews (excluding standard English stop words).

In [ ]:
import re
from collections import Counter

# Standard English stopwords
stopwords = set([
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours",
    "he", "him", "his", "she", "her", "hers", "it", "its", "they", "them", "their",
    "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are",
    "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does",
    "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as", "until",
    "while", "of", "at", "by", "for", "with", "about", "against", "between", "into", "through",
    "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", "out",
    "on", "off", "over", "under", "again", "further", "then", "once", "here", "there",
    "when", "where", "why", "how", "all", "any", "both", "each", "few", "more", "most",
    "other", "some", "such", "no", "nor", "not", "only", "own", "same", "so", "than",
    "too", "very", "s", "t", "can", "will", "just", "don", "should", "now", "ve", "ll",
    "d", "m", "re", "product", "skin", "use", "used", "like", "using"
])

words = []
for text in sample_10k["review_text"].dropna():
    cleaned = re.findall(r"\b[a-zA-Z]{3,}\b", text.lower())
    words.extend([w for w in cleaned if w not in stopwords])

top_words = Counter(words).most_common(20)
words_df = pd.DataFrame(top_words, columns=["Term", "Frequency"])
print("Top 20 Frequent Terms in Customer Reviews:")
display(words_df)

## 9. Exploration Summary & Next Steps

### Pipeline Outputs Summary:
- **Processed Sample:** `InSight_ML/data/processed/cosmetics/cosmetics_10k.csv`
- **Visualizations:** Saved to `InSight_ML/outputs/`
- **Report:** Detailed markdown report available in `InSight_ML/reports/exploration_report.md`

### Downstream Modeling Recommendations:
1. **Class Imbalance Handling**: As positive reviews constitute ~82% of the dataset, subsequent classification models should evaluate class weighting or focal loss.
2. **Weak Labels Verification**: Since weak sentiment labels are inferred from star ratings, future stages should test agreement against actual text sentiment (e.g., reviews with 4 stars complaining of irritation, or 3-star reviews with mixed feedback).